# Load Predictive Models

In [1]:
from nlp_utils import * 
df_train = open_data('./data/liar-plus/train2.tsv')
df_test = open_data('./data/liar-plus/test2.tsv')
df_val = open_data('./data/liar-plus/val2.tsv')
df_train.head()

df_train["statement"] = df_train["statement"].astype(str)
df_train.dropna(subset=['statement','label'], inplace=True)
df_train["statement"].apply(clean_text)
df_train = df_train[['statement','label']]

df_val["statement"] = df_val["statement"].astype(str)
df_val.dropna(subset=['statement','label'], inplace=True)
df_val["statement"].apply(clean_text)
df_val = df_val[['statement','label']]

df_test["statement"] = df_test["statement"].astype(str)
df_test.dropna(subset=['statement','label'], inplace=True)
df_test["statement"].apply(clean_text)
df_test = df_test[['statement','label']]

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import StandardScaler
import torch
import pandas as pd

spam_model_name = "mrm8488/bert-tiny-finetuned-sms-spam-detection"
spam_tokenizer = AutoTokenizer.from_pretrained(spam_model_name)
spam_model = AutoModelForSequenceClassification.from_pretrained(spam_model_name)
spam_model.eval()

def get_spam_scores(text_list, batch_size=16):
    scores = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        inputs = spam_tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=512
        )
        with torch.no_grad():
            outputs = spam_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            scores.extend(probs[:, 1].tolist())
    return scores

df_train['spam_score'] = get_spam_scores(df_train['statement'].tolist())
df_test['spam_score'] = get_spam_scores(df_test['statement'].tolist())
df_val['spam_score'] = get_spam_scores(df_val['statement'].tolist())

spam_scaler = StandardScaler()
df_train['spam_score'] = spam_scaler.fit_transform(df_train[['spam_score']])
df_val['spam_score'] = spam_scaler.transform(df_val[['spam_score']])   
df_test['spam_score'] = spam_scaler.transform(df_test[['spam_score']])

In [3]:
import spacy
# spacy.cli.download("en_core_web_md")
datum = df_train.iloc[0]
nlp = spacy.load("en_core_web_md")
doc = nlp(datum['statement'])
doc.vector.shape
statistic_types = {'CARDINAL', 'PERCENT', 'MONEY', 'QUANTITY'}

def stat_counter(text):
    if not isinstance(text, str):
        return 0
    doc = nlp(text)
    counter = 0
    for ent in doc.ents: 
        if ent.label_ in statistic_types:
            counter += 1
    return counter

from rapidfuzz import fuzz
conservative_bigrams = pd.read_csv('top_conservative_bigrams.csv')['bigram']
liberal_bigrams = pd.read_csv('top_liberal_bigrams.csv')['bigram']
def match_counter(statement, bigram_list, threshold):
    stat = nlp(str(statement))
    word = [word.text.lower() for word in stat]
    bigram_coll = [''.join(word[i:i+2]) for i in range(len(word)-1)]
    matches = 0
    for bigram in bigram_coll:
        for check in bigram_list:
            if fuzz.ratio(bigram, check) >= threshold:
                matches += 1
                break

    return matches


df_train['statistic_count'] = df_train['statement'].apply(stat_counter)
df_train['conservative_bigram_count'] = df_train['statement'].apply(
    lambda x: match_counter(x, conservative_bigrams, threshold=70)
)
df_train['liberal_bigram_count'] = df_train['statement'].apply(
    lambda x: match_counter(x, liberal_bigrams, threshold=70)
)

df_val['statistic_count'] = df_val['statement'].apply(stat_counter)
df_val['conservative_bigram_count'] = df_val['statement'].apply(
    lambda x: match_counter(x, conservative_bigrams, threshold=70)
)
df_val['liberal_bigram_count'] = df_val['statement'].apply(
    lambda x: match_counter(x, liberal_bigrams, threshold=70)
)

df_test['statistic_count'] = df_test['statement'].apply(stat_counter)
df_test['conservative_bigram_count'] = df_test['statement'].apply(
    lambda x: match_counter(x, conservative_bigrams, threshold=70)
)
df_test['liberal_bigram_count'] = df_test['statement'].apply(
    lambda x: match_counter(x, liberal_bigrams, threshold=70)
)

count_features = ["statistic_count", "conservative_bigram_count", "liberal_bigram_count"]
scaler_counts = StandardScaler()

df_train[count_features] = scaler_counts.fit_transform(df_train[count_features])
df_val[count_features]   = scaler_counts.transform(df_val[count_features])
df_test[count_features]  = scaler_counts.transform(df_test[count_features])

In [4]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def emotional_intensity_vader(text):
    if not isinstance(text, str):
        return 0.0
    if len(text) == 0:
        return 0.0
    vs = analyzer.polarity_scores(text)
    return abs(vs['compound'])

for df in [df_train, df_test, df_val]:
    df['emotional_intensity'] = df['statement'].apply(emotional_intensity_vader)

scaler_vader = StandardScaler()
df_train["emotional_intensity"] = scaler_vader.fit_transform(df_train[["emotional_intensity"]])
df_val["emotional_intensity"]   = scaler_vader.transform(df_val[["emotional_intensity"]])
df_test["emotional_intensity"]  = scaler_vader.transform(df_test[["emotional_intensity"]])

In [5]:
def add_veracity_features(df):
    df = df.copy()

    # ----- Spam feature -----
    df['spam_score'] = get_spam_scores(df['statement'].tolist())
    df['spam_score'] = spam_scaler.transform(df[['spam_score']])

    # ----- Count-based features -----
    df['statistic_count'] = df['statement'].apply(stat_counter)
    df['conservative_bigram_count'] = df['statement'].apply(
        lambda x: match_counter(x, conservative_bigrams, threshold=70)
    )
    df['liberal_bigram_count'] = df['statement'].apply(
        lambda x: match_counter(x, liberal_bigrams, threshold=70)
    )
    count_features = ["statistic_count", "conservative_bigram_count", "liberal_bigram_count"]
    df[count_features] = scaler_counts.transform(df[count_features])  # use transform, not fit_transform!

    # ----- Emotional intensity -----
    df['emotional_intensity'] = df['statement'].apply(emotional_intensity_vader)
    df['emotional_intensity'] = scaler_vader.transform(df[['emotional_intensity']])

    return df

In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_train['label'] = le.fit_transform(df_train['label'])
df_val['label'] = le.transform(df_val['label'])
df_test['label'] = le.transform(df_test['label'])

# Load and augment test_data

In [15]:
incoming_df = pd.read_csv('data/labeled_articles.csv')

incoming_text = incoming_df[['id','text']]

incoming_scores = incoming_df[[c for c in incoming_df.columns if c not in set(['id','text'])]]

# GenAI

In [16]:
import pandas as pd
import os
from google import genai
from google.genai import types
import yaml

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

api_key = config["gemini"]["api"]
client = genai.Client(api_key=api_key)

system_prompt = """You are an AI assistant assigned to evaluate the factuality of news statements using a generative fact-checking pipeline.

Your task is to analyze article text, incorporate predictive model outputs WITHOUT overweighting them, and compute factor scores using the scoring recipes below.

==========================
ANTI-BIAS CONSTRAINT
==========================
- Treat predictive model scores only as auxiliary context.
- Do NOT give these predictive scores undue weight.
- If your analysis of the article text contradicts the predictive scores, rely on the TEXTUAL EVIDENCE and explain the discrepancy.

==========================
FACTUALITY FACTORS (6 TOTAL)
==========================

1. AUTHENTICITY  
- Definition: Does the text present evidence that the claims are genuine, verifiable, and traceable?  
- Scoring Recipe (1–10): Look for verifiable details, named sources, timestamps, data, official statements; higher when concrete and falsifiable.  
- Output: numeric score + 1–2 sentences referencing evidence.

2. SENSATIONALISM  
- Definition: Presence of hyperbole, emotional language, exaggeration.  
- Scoring Recipe (1–10): Extract emotional/hyperbolic language, count dramatic constructions, score based on density and prominence.  
- Output: score + 2 example phrases.

3. POLITICAL BIAS  
- Definition: Degree to which the article leans left, center, or right.  
- Scoring Recipe (0–10 + tag): Identify partisan framing or selective omission.  
- Output: numeric score + category {{left, centrist, right, mixed}} + examples.

4. TOXICITY  
- Definition: Hostile, demeaning, or aggressive language.  
- Scoring Recipe (1–10): Identify insults, threats, aggression, and target.  
- Output: score + most toxic example phrase.

5. CONFIRMATION BIAS  
- Definition: Selective presentation of information reinforcing a preferred conclusion.  
- Scoring Recipe (1–10): Identify cherry-picked evidence or missing counterarguments.  
- Output: score + 1 example.

6. SHORT-TERM UTILITY (Profit Incentive)  
- Definition: Degree content maximizes clicks or engagement.  
- Scoring Recipe (1–10): Detect clickbait, urgent calls to action, monetization cues.  
- Output: score + 1–2 indicators of profit-driven framing.

==========================
VERACITY LABEL (REQUIRED)
==========================
- Output a final factuality classification: pants-fire, false, barely-true, half-true, mostly-true, true

==========================
OUTPUT FORMAT
==========================
- Final veracity label  
- Scores for all six factors  
- Short paragraph explaining WHY each score was assigned  
- Final combined veracity explanation  
- Structured JSON-like block per statement

==========================
REASONING FORMAT
==========================
- Provide concise, structured rationale per factor: key textual evidence, weighting of predictive scores, numeric reasoning (2–4 bullet points).  
- DO NOT reveal internal chain-of-thought or hidden reasoning.

==========================
EXAMPLES
==========================
Here are some examples of the expected output format and reasoning structure:
    - Example 1:
    {{
    "article": "A new study shows that drinking green tea daily reduces the risk of heart disease by 30%. The study surveyed 10,000 adults over 5 years and was published in the Journal of Cardiology.",
    "author": "Health Daily News",
    "authenticity": "9",    # The study is published in a reputable journal and includes a large sample size and clear methodology.
    "sensationalism": "3",  # Mild exaggeration in phrasing 'reduces the risk by 30%', but mostly factual.
    "political bias": "1",  # No political framing detected.
    "toxicity": "1",    # No hostile or demeaning language.
    "confirmation bias": "2",   # Some emphasis on positive findings without mention of limitations, but minimal.
    "short-term utility": "2",  # Headline is slightly attention-grabbing, but content is largely informational.
    }}

    - Example 2:
    {{
        "article": "Politician X is the worst leader in history! Everything they touch fails, and the economy is collapsing under their rule.",
        "author": "Partisan Weekly",
        "authenticity": "2",    # Claims are vague and unsupported; no verifiable evidence is cited.
        "sensationalism": "9",  # Highly emotional and hyperbolic language such as 'worst leader' and 'everything they touch fails'.
        "political bias": "10", # Strong right/left framing (depending on context); clearly targeting a political figure.
        "toxicity": "8",    # Personal attacks and extreme language directed at the politician.
        "confirmation bias": "8",   # Selectively presents negative information; ignores any positive actions or counterarguments.
        "short-term utility": "7",  # Designed to provoke outrage and drive engagement.
    }}

    - Example 3:
    {{
        "article": "Local bakery wins award for best chocolate cake. The contest included 50 bakeries, and judges highlighted creativity and flavor balance.",
        "author": "Town Gazette",
        "authenticity": "8",    # Contest results are verifiable; named judges and participants are listed.
        "sensationalism": "2",  # Mildly positive tone but not exaggerated.
        "political bias": "1",  # No political content.
        "toxicity": "1",    # No toxic language present.
        "confirmation bias": "1",   # Balanced reporting; no selective framing detected.
        "short-term utility": "3",  # Lightly engaging human-interest story.
    }}

    - Example 4:
    {{
        "article": "Experts warn that a major cyberattack could hit the US next month. Anonymous sources say the threat is imminent.",
        "author": "TechAlert News",
        "authenticity": "3", # Based on anonymous sources; lacks verifiable evidence.
        "sensationalism": "8", # Phrases like 'major cyberattack' and 'imminent' are highly alarming.
        "political bias": "2", # No clear political slant, mostly technical framing.
        "toxicity": "1", # No hostile language.
        "confirmation bias": "5", # Focuses on worst-case scenarios without context or probability discussion.
        "short-term utility": "8", # Headline and framing likely to generate clicks and engagement.
    }}
"""

user_prompt = f""""Here is the dataset to evaluate:
        {incoming_text}
Here are the predictive model outputs:
        {incoming_scores}
Begin the factor scoring and veracity prediction.
"""

response = client.models.generate_content(
    model="gemini-2.5-pro",
    config=types.GenerateContentConfig(
        system_instruction=system_prompt),
    contents=user_prompt
)

print(response.text)

Here are the evaluations for each news statement:

### Statement 1
**VERACITY: FALSE**
This statement is satirical. The claim that a mayor-elect would ban all cars and force citizens to walk from another state is an absurd premise designed for comedic effect, not to be taken as a factual report.

*   **AUTHENTICITY: 1**
    *   **Rationale:** The claim is patently absurd and not verifiable. The entire premise is a fabrication for satirical purposes, lacking any connection to real-world policy or events. The model's low score of 2 aligns with this assessment.
*   **SENSATIONALISM: 8**
    *   **Rationale:** The text uses hyperbolic and exaggerated scenarios for comedic effect. Phrases like "ban cars from the city" and "forcing all citizens to walk the last mile from New Jersey" are extreme overstatements.
*   **POLITICAL BIAS: 6 | mixed**
    *   **Rationale:** The article is political satire targeting a specific, real-life progressive politician (Zohran Mamdani). It uses his political 

In [10]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""pants-on-fire
Political Bias: right-leaning
Sensationalism: large sensationalism
Spam: heavy spam
mostly-true
Political Bias: left-leaning
Sensationalism: moderate sensationalism
Spam: not spam
half-true
Political Bias: left-leaning
Sensationalism: moderate sensationalism
Spam: not spam
true
Political Bias: centrist
Sensationalism: no sensationalism
Spam: not spam
false
Political Bias: right-leaning
Sensationalism: moderate sensationalism
Spam: not spam
barely-true
Political Bias: right-leaning
Sensationalism: moderate sensationalism
Spam: not spam
pants-on-fire
Political Bias: right-leaning
Sensationalism: large sensationalism
Spam: not spam
mostly-true
Political Bias: left-leaning
Sensationalism: no sensationalism
Spam: not spam
half-true
Political Bias: right-leaning
Sensationalism: moderate sensationalism
Spam: no

In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd

spam_model_name = "mrm8488/bert-tiny-finetuned-sms-spam-detection"
spam_tokenizer = AutoTokenizer.from_pretrained(spam_model_name)
spam_model = AutoModelForSequenceClassification.from_pretrained(spam_model_name)
spam_model.eval()

def get_spam_scores(text_list, batch_size=16):
    scores = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        inputs = spam_tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=512
        )
        with torch.no_grad():
            outputs = spam_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            scores.extend(probs[:, 1].tolist())
    return scores

test_df['spam_score'] = get_spam_scores(test_df['statement'].to_list())

In [12]:
test_df

,index,id,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context,justification,spam_score
0,1,238.json,"When Obama was sworn into office, he DID NOT u...","obama-birth-certificate,religion",chain-email,NaN,NaN,none,11,43,8,5,105,NaN,Ellison used a Koran that once belonged to Tho...,0.065085
1,2,7891.json,Says Having organizations parading as being so...,"campaign-finance,congress,taxes",earl-blumenauer,U.S. representative,Oregon,democrat,0,1,1,1,0,a U.S. Ways and Means hearing,"However, we have two professors who say the la...",0.087582
2,3,8169.json,Says nearly half of Oregons children are poor.,poverty,jim-francesconi,Member of the State Board of Higher Education,Oregon,none,0,1,1,1,0,an opinion article,"In fact, if you use federal definitions for po...",0.064704
3,4,929.json,On attacks by Republicans that various program...,"economy,stimulus",barack-obama,President,Illinois,democrat,70,71,160,163,9,interview with CBS News,Obama's point is that some perspective is in o...,0.112518
4,5,9416.json,Says when armed civilians stop mass shootings ...,guns,jim-rubens,Small business owner,New Hampshire,republican,1,1,0,1,0,"in an interview at gun shop in Hudson, N.H.",Rubens said when armed civilians stop mass sho...,0.076772
5,6,6861.json,Says Tennessee is providing millions of dollar...,"education,state-budget",andy-berke,Lawyer and state senator,Tennessee,democrat,0,0,0,0,0,a letter to state Senate education committee c...,"However, even Huffman, who is very sympathetic...",0.072836
6,7,1122.json,The health care reform plan would set limits s...,health-care,club-growth,NaN,NaN,none,4,5,4,2,0,a TV ad,"So, back to the Club for Growth ad. There is n...",0.174156
7,8,13138.json,Says Donald Trump started his career back in 1...,"candidates-biography,diversity,housing",hillary-clinton,Presidential candidate,New York,democrat,40,29,69,76,7,the first presidential debate,"Clinton said that Trump ""started his career ba...",0.081957
8,9,1880.json,Bill White has a long history of trying to lim...,military,republican-party-texas,NaN,Texas,republican,3,1,1,3,1,an e-mail,Did White's positions which he hasn't backed ...,0.066932
9,10,12803.json,John McCains chief economic adviser during the...,economy,tim-kaine,U.S. Senator,Virginia,democrat,8,3,15,15,0,a speech at the Democratic National Convention...,"Kaine said ""John McCains chief economic advise...",0.199033
